# MaisonDeLUX — nationwide sale-listing pipeline V3

Run all cells. The default `PILOT` mode is safe and bounded. `FAST` and `FULL` automatically rerun the pilot first and stop before creating `maisondelux_clean_v3` unless every release gate passes. Existing `maisondelux_clean.csv` is never overwritten.

## 1. Configuration

In [ ]:
from pathlib import Path
import json

def find_project_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / 'config' / 'scraping_v3.json').exists():
            return candidate
    raise FileNotFoundError('Could not locate config/scraping_v3.json')

PROJECT_ROOT = find_project_root(Path.cwd())
MODE = 'PILOT'  # PILOT (~500), FAST (~20k), or FULL (up to ~50k when available)
CONFIG_PATH = PROJECT_ROOT / 'config' / 'scraping_v3.json'
FORCE_PILOT_GATE = True
PROJECT_ROOT, MODE

## 2. Imports

In [ ]:
import sys
import pandas as pd
from IPython.display import display, Markdown

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from ml.src.scraping_v3 import PipelineConfig, V3_COLUMNS, run_pipeline
from ml.src.scraping_v3.geography import REGION_CITIES
from ml.src.scraping_v3.pipeline import summary_tables

## 3. Paths

In [ ]:
settings = json.loads(CONFIG_PATH.read_text(encoding='utf-8'))
paths = {name: PROJECT_ROOT / value for name, value in settings['paths'].items()}
pd.Series({name: str(path) for name, path in paths.items()}, name='portable project path')

## 4. Moroccan geography reference

In [ ]:
geography_reference = pd.DataFrame([
    {'region': region, 'configured_cities': len(cities), 'cities': ', '.join(cities)}
    for region, cities in REGION_CITIES.items()
])
display(geography_reference)
assert len(geography_reference) == 12

## 5. Source configuration

In [ ]:
source_configuration = pd.DataFrame(settings['sources'])
display(source_configuration[['name', 'kind', 'enabled', 'authorization_reference', 'policy_note']])
display(Markdown('Live listing requests require both `enabled: true` and a documented `authorization_reference`. Licensed CSV, Parquet, JSON, and JSONL feeds belong under `data/input/authorized/`.'))

## 6. Scraping / collection

In [ ]:
result = run_pipeline(PipelineConfig(
    mode=MODE,
    config_path=CONFIG_PATH,
    project_root=PROJECT_ROOT,
    force_pilot_gate=FORCE_PILOT_GATE,
))
display(pd.Series(result.source_statuses, name='status'))

## 7. Checkpoint / resume

In [ ]:
checkpoint_files = sorted(paths['checkpoints'].glob('*.jsonl'))
display(pd.DataFrame({'checkpoint': [str(path.relative_to(PROJECT_ROOT)) for path in checkpoint_files], 'bytes': [path.stat().st_size for path in checkpoint_files]}))
assert result.report['acceptance_checks']['checkpoint_resume_tested']

## 8. Normalization

In [ ]:
clean = pd.read_csv(result.processed_path)
assert list(clean.columns) == V3_COLUMNS
display(clean.head(5))

## 9. Property-type validation

In [ ]:
tables = summary_tables(result)
display(tables['property_type'])
contradictions = clean[clean['validation_reasons'].fillna('').str.contains('property_type_contradiction', regex=False)]
display(contradictions[['title_raw', 'property_type', 'validation_reasons']].head(20))

## 10. Location validation

In [ ]:
display(tables['region'])
display(tables['city'])
display(clean[['region', 'city', 'neighborhood', 'location_raw']].sample(min(20, len(clean)), random_state=42))

## 11. Deduplication

In [ ]:
display(pd.Series(result.report['distributions']['deduplication_status'], name='count'))
assert clean['listing_id'].is_unique

## 12. Quality validation

In [ ]:
display(tables['acceptance'])
display(tables['missing_percent'])
display(pd.DataFrame.from_dict(result.report['publication_date_by_source'], orient='index'))
if MODE == 'PILOT' and not result.pilot_passed:
    failed = tables['acceptance'].index[~tables['acceptance']['passed']].tolist()
    display(Markdown('**Pilot is intentionally NOT approved for scaling.** Failed gates: ' + ', '.join(failed)))

## 13. Dataset summary

In [ ]:
display(tables['summary'])
display(tables['source'])
display(clean.groupby(['source', 'region', 'city', 'property_type'], dropna=False).size().rename('listings').reset_index().sort_values('listings', ascending=False).head(50))

## 14. Export

In [ ]:
exports = {
    'processed_csv': result.processed_path,
    'processed_parquet': result.processed_path.with_suffix('.parquet'),
    'quality_report_json': result.report_json_path,
    'quality_report_markdown': result.report_markdown_path,
}
display(pd.Series({name: str(path) for name, path in exports.items()}, name='output'))
if MODE in {'FAST', 'FULL'}:
    assert result.pilot_passed is None or result.pilot_passed
    display(Markdown('V3 final export completed without modifying `maisondelux_clean.csv`.'))
else:
    display(Markdown('Pilot artifacts were exported. Set `MODE = FAST` only after all pilot gates pass.'))